# MURE + Sentence-based LLM 3B (1-Click Auto-Run) 🚀🧠

ဒီ Notebook က GitHub ကနေ Code တွေကို အလိုအလျောက် ဆွဲယူပြီး Google Drive ပေါ်က (`svo cc brain`) Knowledge Base နဲ့ချိတ်ဆက်ပေးပါမယ်။

**လုပ်ဆောင်ရမယ့်အဆင့်:**
1. မျက်နှာပြင်အပေါ်က `Runtime` ကိုနှိပ်ပါ။
2. `Run All` (Ctrl+F9) ကိုနှိပ်လိုက်ရုံပါပဲ!

> **မှတ်ချက်:** GitHub Repository URL ကို သင့်ကိုယ်ပိုင် URL နဲ့ ပြောင်းပေးဖို့ `Cell 2` မှာ ပြင်နိုင်ပါတယ်။

In [ ]:
# 1. Google Drive ကိုချိတ်ဆက်မယ် (Knowledge Base နဲ့ Priming Data ယူရန်အတွက်)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. GitHub ကနေ နောက်ဆုံးထွက် Code တွေကို ဆွဲယူမယ်
import os
import shutil

# ⚠️ သင့်ရဲ့ GitHub Repository Link ကို ဒီနေရာမှာ ထည့်ပါ
GITHUB_REPO_URL = "https://github.com/your-username/your-repo-name.git"

PROJECT_FOLDER = "/content/mure_project"

if os.path.exists(PROJECT_FOLDER):
    shutil.rmtree(PROJECT_FOLDER) # အသစ်ဆွဲဖို့ အဟောင်းဖျက်မယ်

!git clone {GITHUB_REPO_URL} {PROJECT_FOLDER}

%cd {PROJECT_FOLDER}

print("\n✅ GitHub မှ Code များ အောင်မြင်စွာ ဆွဲယူပြီးပါပြီ။")

# Install dependencies if any
if os.path.exists("requirements.txt"):
    !pip install -r requirements.txt -q

In [ ]:
# 3. Google Drive / MyDrive / svo cc brain သို့ချိတ်ဆက်မှုနဲ့ Data Extract လုပ်ခြင်း
!python setup_colab_env.py

In [ ]:
# 4. MURE + 3B LLM စနစ်ကို API (Ngrok) ခံပြီး run ခြင်း
# React Web UI နဲ့ ချိတ်ဆက်ဖို့အတွက် Ngrok Auth Token ကို ဒီနေရာမှာ ထည့်ပါ။
NGROK_TOKEN = 'YOUR_NGROK_AUTH_TOKEN_HERE'  # <--- ဒီနေရာမှာ Token ပြောင်းထည့်ပါ

import os
os.environ['NGROK_AUTH_TOKEN'] = NGROK_TOKEN

# Install missing packages if needed
!pip install fastapi uvicorn pydantic pyngrok nest-asyncio -q

from pyngrok import ngrok
import uvicorn
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# We import the LLM architecture from run_mure_colab
from run_mure_colab import mure, MURESentenceLLM

if not NGROK_TOKEN or NGROK_TOKEN == 'YOUR_NGROK_AUTH_TOKEN_HERE':
    print('❌ NGROK_TOKEN ကို ထည့်ပေးပါ။')
else:
    app = FastAPI(title='MURE API Backend (MURE + Sentence LLM)')
    app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

    class ChatRequest(BaseModel):
        message: str

    llm = MURESentenceLLM(mure_reasoner=mure)

    @app.get('/health')
    def health_check():
        return {'status': 'online', 'version': 'MURE + Sentence LLM'}

    @app.get('/stats')
    def get_stats():
        return {'causalRules': len(mure.rules)}

    @app.post('/chat')
    def chat(req: ChatRequest):
        try:
            reply = llm.generate_response(req.message)
            return {
                'reply': reply,
                'frame': {'effect': 'Sentence LLM generated'},
                'learned': False,
                'source': 'mure_3b_sentence_llm',
                'stats': {'causalRules': len(mure.rules)}
            }
        except Exception as e:
            raise HTTPException(status_code=500, detail=str(e))

    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8000)
    print('==========================================================')
    print(f'⭐ PUBLIC API URL: {public_url.public_url} ⭐')
    print('ဒီ URL ကို copy ကူးပြီး React Web UI ရဲ့ Settings ထဲက Python Backend API URL နေရာမှာ ထည့်ပါ။')
    print('==========================================================')

    nest_asyncio.apply()
    uvicorn.run(app, host='0.0.0.0', port=8000)
